# Confusion matrices (paper Figures 7–9)

> **Note:** this notebook rebuilds the confusion matrices from the **raw per-iteration
> model outputs** (`results/<model>/raw/` and `appendix/results/american/<model>/raw/`),
> which are **not shipped with the repository** (they are regenerated by the pipeline
> notebooks). Without them every model config is skipped (`SKIP: ... not found`).
> The matrices themselves are already provided under `results/confusion_matrix/` and
> `appendix/results/confusion_matrix/`.
>
> Step 1.2 matrices display **per-run average counts** (raw counts ÷ 15 runs = 3
> techniques × 5 iterations), as in the paper; Steps 3–4 display raw counts.


## Setup & configuration

Imports, factor-code definitions, the model → raw-path mapping, and output routing:
Japanese-prompt matrices go to `results/confusion_matrix/` (paper Figures 7–9), American-persona
matrices to `appendix/results/confusion_matrix/`.

In [1]:
import os, json, re, glob, unicodedata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from collections import defaultdict

In [2]:
# ============================================================
# Configuration
# ============================================================
# Factor definitions
FACTOR_DEFS = {
    'A1': 'Restaurant Quality',
    'A2': 'Accessibility & Location',
    'A3': 'Schedule Constraints',
    'A4': 'Social Utility for Consensus',
    'A5': 'Inertia',
    'A6': 'Economic Considerations',
    'A7': 'Others',
}

# Resolve the repository root regardless of where Jupyter was started
# (the kernel's cwd is this notebook's folder when opened from evaluation/)
if not os.path.isdir('data') and os.path.isdir('../data'):
    os.chdir('..')
WORK_DIR = '.'  # repository root
DATA_DIR = os.path.join(WORK_DIR, 'data')
GOLD_DIR = os.path.join(DATA_DIR, 'gold')

MODEL_CONFIGS = {
    'Japanese_GPT-5':     os.path.join(WORK_DIR, 'results/gpt5/raw'),
    'Japanese_GPT-4o_t0': os.path.join(WORK_DIR, 'results/gpt4o/raw'),
    'American_GPT-5':     os.path.join(WORK_DIR, 'appendix/results/american/gpt5/raw'),
    'American_GPT-4o_t0': os.path.join(WORK_DIR, 'appendix/results/american/gpt4o/raw'),
}

OUTPUT_DIRS = {
    'Japanese': os.path.join(WORK_DIR, 'results/confusion_matrix'),
    'American': os.path.join(WORK_DIR, 'appendix/results/confusion_matrix'),
}
for _d in OUTPUT_DIRS.values():
    os.makedirs(_d, exist_ok=True)

def normalize_name(x):
    return str(x).strip().lower()

def parse_factors(s):
    if not s or str(s).strip().lower() == 'none':
        return set()
    return {f.strip().lower() for f in str(s).split(',') if f.strip() and f.strip().lower() != 'none'}

print(f'WORK_DIR: {WORK_DIR}')
print(f'Models: {list(MODEL_CONFIGS.keys())}')

WORK_DIR: .
Models: ['Japanese_GPT-5', 'Japanese_GPT-4o_t0', 'American_GPT-5', 'American_GPT-4o_t0']


## Plotting Functions

In [3]:
def plot_general_cm(crosstab_df, title, filepath):
    """Steps 1.2 / 3 confusion matrix: overall percentages, bold outer border."""
    rows, cols = crosstab_df.shape
    if rows == 0 or cols == 0:
        print(f"  SKIP: '{title}' - no data")
        return

    total = crosstab_df.values.sum()
    if total == 0:
        total = 1
    pct_df = crosstab_df / total * 100

    # Normalize for color: overall (0-max_pct)
    max_val = crosstab_df.values.max()
    if max_val == 0:
        max_val = 1
    norm_df = crosstab_df / max_val

    cell_size = 1.6
    figsize_w = cols * cell_size + 2.5
    figsize_h = rows * cell_size + 2.0

    fig, ax = plt.subplots(figsize=(figsize_w, figsize_h))

    cmap = plt.cm.Blues

    for i in range(rows):
        for j in range(cols):
            val = norm_df.iloc[i, j]
            color = cmap(val * 0.85 + 0.05)
            rect = plt.Rectangle((j, rows - 1 - i), 1, 1, facecolor=color,
                                  edgecolor='#cccccc', linewidth=0.5)
            ax.add_patch(rect)

            count = crosstab_df.iloc[i, j]
            pct = pct_df.iloc[i, j]
            text_color = 'white' if val > 0.55 else 'black'

            count_txt = f'{count:.2f}' if isinstance(count, float) else f'{count}'
            ax.text(j + 0.5, rows - 1 - i + 0.58, count_txt,
                    ha='center', va='center', fontsize=12, fontweight='bold',
                    color=text_color)
            ax.text(j + 0.5, rows - 1 - i + 0.30, f'({pct:.1f}%)',
                    ha='center', va='center', fontsize=10,
                    color=text_color)

    # Thin internal lines
    for i in range(1, rows):
        ax.plot([0, cols], [i, i], color='#cccccc', linewidth=0.5)
    for j in range(1, cols):
        ax.plot([j, j], [0, rows], color='#cccccc', linewidth=0.5)

    # Bold outer border only
    for spine_side in ['top', 'bottom', 'left', 'right']:
        ax.spines[spine_side].set_visible(False)
    border = plt.Rectangle((0, 0), cols, rows, fill=False,
                            edgecolor='black', linewidth=2.5)
    ax.add_patch(border)

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_xticks([j + 0.5 for j in range(cols)])
    ax.set_xticklabels([str(c).title() for c in crosstab_df.columns], fontsize=12, fontweight='bold')
    ax.set_yticks([rows - 1 - i + 0.5 for i in range(rows)])
    ax.set_yticklabels([str(r).title() for r in crosstab_df.index], fontsize=12, fontweight='bold')

    ax.set_xlabel('Predicted', fontsize=13, fontweight='bold', labelpad=10)
    ax.set_ylabel('Ground Truth', fontsize=13, fontweight='bold', labelpad=10)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.tick_params(length=0)
    ax.set_aspect('equal')

    # Colorbar (count scale)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=crosstab_df.values.max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=10)

    plt.tight_layout(pad=1.2)
    plt.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    # SVG output
    svg_path = filepath.rsplit('.', 1)[0] + '.svg'
    plt.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  Saved: {filepath}")
    print(f"  Saved: {svg_path}")


def plot_step4_cm(crosstab_df, title, filepath):
    """Step 4 confusion matrix: row-wise percentages, bold row separators."""
    rows, cols = crosstab_df.shape
    if rows == 0 or cols == 0:
        print(f"  SKIP: '{title}' - no data")
        return

    # Row percentages
    row_sums = crosstab_df.sum(axis=1).replace(0, 1)
    pct_df = crosstab_df.div(row_sums, axis=0) * 100

    # Normalize for color: row-wise (each row 0-1)
    norm_df = pct_df / 100.0

    cell_w = 1.55
    cell_h = 1.15
    figsize_w = cols * cell_w + 3.0
    figsize_h = rows * cell_h + 4.8  # extra space for factor defs

    fig, ax = plt.subplots(figsize=(figsize_w, figsize_h))

    cmap = plt.cm.Blues

    # Draw cells
    for i in range(rows):
        for j in range(cols):
            val = norm_df.iloc[i, j]
            color = cmap(val * 0.85 + 0.05)
            rect = plt.Rectangle((j, rows - 1 - i), 1, 1, facecolor=color,
                                  edgecolor='#cccccc', linewidth=0.5)
            ax.add_patch(rect)

            count = crosstab_df.iloc[i, j]
            pct = pct_df.iloc[i, j]
            text_color = 'white' if val > 0.55 else 'black'

            ax.text(j + 0.5, rows - 1 - i + 0.60, f'{count}',
                    ha='center', va='center', fontsize=11.5, fontweight='bold',
                    color=text_color)
            ax.text(j + 0.5, rows - 1 - i + 0.32, f'({pct:.1f}%)',
                    ha='center', va='center', fontsize=9.5,
                    color=text_color)

    # Bold horizontal lines between rows (row-wise boundary)
    for i in range(rows + 1):
        ax.plot([0, cols], [i, i], color='black', linewidth=2.2)
    # Thin vertical lines
    for j in range(cols + 1):
        ax.plot([j, j], [0, rows], color='#999999', linewidth=0.8)
    # Bold outer vertical borders
    ax.plot([0, 0], [0, rows], color='black', linewidth=2.2)
    ax.plot([cols, cols], [0, rows], color='black', linewidth=2.2)

    # Axis labels
    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_xticks([j + 0.5 for j in range(cols)])
    ax.set_xticklabels([str(c).upper() for c in crosstab_df.columns],
                       fontsize=11.5, fontweight='bold', rotation=0)
    ax.set_yticks([rows - 1 - i + 0.5 for i in range(rows)])
    ax.set_yticklabels([str(r).upper() for r in crosstab_df.index],
                       fontsize=11.5, fontweight='bold')

    ax.set_xlabel('Predicted', fontsize=13, fontweight='bold', labelpad=10)
    ax.set_ylabel('Ground Truth', fontsize=13, fontweight='bold', labelpad=10)
    ax.set_title(title, fontsize=15, fontweight='bold', pad=15)
    ax.tick_params(length=0)

    # Colorbar (row percentage 0–100%)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=crosstab_df.values.max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=10)

    # Factor definitions below — stacked vertically for readability
    factor_lines = []
    for code in ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7']:
        factor_lines.append(f"{code}: {FACTOR_DEFS[code]}")
    # 2 rows: 4 + 3
    line1 = '    '.join(factor_lines[:4])
    line2 = '    '.join(factor_lines[4:])
    factor_text = line1 + '\n' + line2

    fig.text(0.5, 0.02, factor_text, ha='center', va='bottom',
             fontsize=9, style='italic', color='#333333',
             family='sans-serif')

    ax.set_aspect('equal')
    plt.tight_layout(rect=[0, 0.10, 1, 1])
    plt.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    # SVG output
    svg_path = filepath.rsplit('.', 1)[0] + '.svg'
    plt.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  Saved: {filepath}")
    print(f"  Saved: {svg_path}")

print("Plotting functions loaded")

Plotting functions loaded


## Collect All Iteration Pairs

In [4]:
def collect_all_pairs(result_dir, gold_dir):
    logs = sorted([d for d in os.listdir(result_dir)
                   if d.endswith('_log') and os.path.isdir(os.path.join(result_dir, d))])

    sugg_pairs, resp_pairs, sent_pairs, factor_pairs = [], [], [], []

    for log_name in logs:
        log_path = os.path.join(result_dir, log_name)

        gold = {}
        for key, fname in [('step1_2', 'step1_2_gold.json'), ('step3', 'step3_gold.json'), ('step4', 'step4_gold.json')]:
            fpath = os.path.join(gold_dir, log_name, fname)
            if os.path.exists(fpath):
                with open(fpath, 'r', encoding='utf-8') as f:
                    gold[key] = json.load(f)

        final_path = os.path.join(log_path, f'{log_name}_final_results.json')
        if not os.path.exists(final_path):
            continue
        with open(final_path, 'r', encoding='utf-8') as f:
            final = json.load(f)
        scope_p = set(final['step1']['participants'])
        scope_r = {normalize_name(r) for r in final['step1']['restaurant_brands']}

        all_files = os.listdir(log_path)

        # Step 1.2
        gold_s12 = gold.get('step1_2', {})
        gold_sugg = {r['participant']: r['suggestion_type'] for r in gold_s12.get('suggestion_table', [])}
        gold_resp = {r['participant']: r['response_type'] for r in gold_s12.get('response_table', [])}

        s1_files = sorted([f for f in all_files if f.startswith('Step1_') and '_normalized_' in f and f.endswith('.json')])
        for s1f in s1_files:
            try:
                s1_data = json.load(open(os.path.join(log_path, s1f)))
                pred_sugg = {r['participant']: r.get('suggestion_type', '') for r in s1_data.get('suggestion_table', [])}
                pred_resp = {r['participant']: r.get('response_type', '') for r in s1_data.get('response_table', [])}
                for p, g in gold_sugg.items():
                    pr = pred_sugg.get(p, '')
                    if g:
                        sugg_pairs.append((g.lower(), pr.lower() if pr else 'missing'))
                for p, g in gold_resp.items():
                    pr = pred_resp.get(p, '')
                    if g:
                        resp_pairs.append((g.lower(), pr.lower() if pr else 'missing'))
            except:
                continue

        # Step 3
        gold_s3 = gold.get('step3', {})
        gold_s3_table = gold_s3.get('perception_table', gold_s3.get('sentiment_table', []))
        gold_s3_map = {}
        for row in gold_s3_table:
            p, r = row.get('participant', ''), normalize_name(row.get('restaurant', ''))
            if p in scope_p and r in scope_r:
                gold_s3_map[(p, r)] = row.get('perception', row.get('sentiment', '')).strip()

        s3_files = sorted([f for f in all_files if f.startswith(f'{log_name}_Step3_') and re.match(r'.*_\d+\.json$', f)])
        for s3f in s3_files:
            try:
                s3_data = json.load(open(os.path.join(log_path, s3f)))
                pred_table = s3_data.get('Result', {}).get('sentiment_table', [])
                pred_s3_map = {}
                for row in pred_table:
                    p, r = row.get('participant', ''), normalize_name(row.get('restaurant', ''))
                    if p in scope_p and r in scope_r:
                        pred_s3_map[(p, r)] = row.get('sentiment', row.get('perception', '')).strip()
                for key, gv in gold_s3_map.items():
                    pv = pred_s3_map.get(key, 'Missing')
                    if gv:
                        sent_pairs.append((gv, pv if pv else 'Missing'))
            except:
                continue

        # Step 4
        gold_s4 = gold.get('step4', {})

        def build_integrated_map(pref_tbl, cons_tbl, pref_key='preferences', cons_key='constraints'):
            m = {}
            for row in pref_tbl:
                p, r = row['participant'], normalize_name(row['restaurant'])
                if p in scope_p and r in scope_r:
                    m.setdefault((p, r), set()).update(parse_factors(row.get(pref_key, 'None')))
            for row in cons_tbl:
                p, r = row['participant'], normalize_name(row['restaurant'])
                if p in scope_p and r in scope_r:
                    m.setdefault((p, r), set()).update(parse_factors(row.get(cons_key, 'None')))
            return m

        gold_int = build_integrated_map(
            gold_s4.get('preference_table', []),
            gold_s4.get('constraint_table', [])
        )
        all_factors_set = {'a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7'}

        s4_files = sorted([f for f in all_files if f.startswith(f'{log_name}_Step4_') and re.match(r'.*_\d+\.json$', f)])
        for s4f in s4_files:
            try:
                s4_data = json.load(open(os.path.join(log_path, s4f)))
                pred_result = s4_data.get('Result', {})
                pred_int = build_integrated_map(
                    pred_result.get('preference_table', []),
                    pred_result.get('constraint_table', [])
                )
                for key, gold_factors in gold_int.items():
                    pred_factors = pred_int.get(key, set())
                    if not gold_factors:
                        continue
                    for gf in gold_factors:
                        if gf not in all_factors_set:
                            continue
                        if pred_factors:
                            for pf in pred_factors:
                                if pf in all_factors_set or pf == 'none':
                                    factor_pairs.append((gf, pf))
                        else:
                            factor_pairs.append((gf, 'none'))
            except:
                continue

    return sugg_pairs, resp_pairs, sent_pairs, factor_pairs

print("Data collection function loaded")

Data collection function loaded


## Generate Confusion Matrix PNGs

In [5]:
for config_label, result_dir in MODEL_CONFIGS.items():
    if not os.path.isdir(result_dir):
        print(f"SKIP: {config_label} ({result_dir} not found)")
        continue

    print(f"\n{'='*60}")
    print(f"  {config_label}")
    print(f"{'='*60}")

    sugg, resp, sent, factors = collect_all_pairs(result_dir, GOLD_DIR)
    print(f"  Pairs: suggestion={len(sugg)}, response={len(resp)}, sentiment={len(sent)}, factors={len(factors)}")

    prefix = config_label
    OUTPUT_DIR = OUTPUT_DIRS['Japanese' if config_label.startswith('Japanese') else 'American']

    # Step 1.2 Suggestion
    if sugg:
        labels = ['strong', 'moderate', 'weak']
        df_s = pd.DataFrame(sugg, columns=['Ground Truth', 'Predicted'])
        ct = pd.crosstab(df_s['Ground Truth'], df_s['Predicted'])
        cols = [c for c in labels if c in ct.columns] + [c for c in sorted(ct.columns) if c not in labels]
        rows = [r for r in labels if r in ct.index]
        ct = ct.reindex(index=rows, columns=cols, fill_value=0)
        ct = ct / 15  # per-run average (3 techniques x 5 iterations), as in the paper
        plot_general_cm(ct, f'Step 1.2 Suggestion ({prefix})',
                        os.path.join(OUTPUT_DIR, f'{prefix}_Step1_2_Suggestion_CM.png'))

    # Step 1.2 Response
    if resp:
        labels = ['agreeable', 'moderate', 'disagreeable']
        df_r = pd.DataFrame(resp, columns=['Ground Truth', 'Predicted'])
        ct = pd.crosstab(df_r['Ground Truth'], df_r['Predicted'])
        cols = [c for c in labels if c in ct.columns] + [c for c in sorted(ct.columns) if c not in labels]
        rows = [r for r in labels if r in ct.index]
        ct = ct.reindex(index=rows, columns=cols, fill_value=0)
        ct = ct / 15  # per-run average (3 techniques x 5 iterations), as in the paper
        plot_general_cm(ct, f'Step 1.2 Response ({prefix})',
                        os.path.join(OUTPUT_DIR, f'{prefix}_Step1_2_Response_CM.png'))

    # Step 3 Sentiment
    if sent:
        labels = ['Mix', 'Negative', 'Neutral', 'Positive']
        df_p = pd.DataFrame(sent, columns=['Ground Truth', 'Predicted'])
        ct = pd.crosstab(df_p['Ground Truth'], df_p['Predicted'])
        rows = [l for l in labels if l in ct.index]
        cols = [l for l in labels if l in ct.columns] + [c for c in sorted(ct.columns) if c not in labels]
        ct = ct.reindex(index=rows, columns=cols, fill_value=0)
        plot_general_cm(ct, f'Step 3. Perception Table ({prefix})',
                        os.path.join(OUTPUT_DIR, f'{prefix}_Step3_Perception_CM.png'))

    # Step 4 Integrated
    if factors:
        factor_labels = ['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7']
        df_f = pd.DataFrame(factors, columns=['Ground Truth', 'Predicted'])
        ct = pd.crosstab(df_f['Ground Truth'], df_f['Predicted'])
        cols = [c for c in factor_labels if c in ct.columns] + (['none'] if 'none' in ct.columns else [])
        for c in sorted(ct.columns):
            if c not in cols:
                cols.append(c)
        rows = [r for r in factor_labels if r in ct.index]
        ct = ct.reindex(index=rows, columns=cols, fill_value=0)
        plot_step4_cm(ct, f'Step 4. Integrated Table ({prefix})',
                      os.path.join(OUTPUT_DIR, f'{prefix}_Step4_Integrated_CM.png'))

# List all generated files
print(f"\n{'='*60}")
for OUTPUT_DIR in OUTPUT_DIRS.values():
    print(f"All PNGs in {OUTPUT_DIR}/:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        if f.endswith('.png'):
            size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
            print(f"  {f} ({size_kb:.0f} KB)")

SKIP: Japanese_GPT-5 (.\results/gpt5/raw not found)
SKIP: Japanese_GPT-4o_t0 (.\results/gpt4o/raw not found)
SKIP: American_GPT-5 (.\appendix/results/american/gpt5/raw not found)
SKIP: American_GPT-4o_t0 (.\appendix/results/american/gpt4o/raw not found)

All PNGs in .\results/confusion_matrix/:
  Japanese_GPT-4o_t0_Step1_2_Response_CM.png (66 KB)
  Japanese_GPT-4o_t0_Step1_2_Suggestion_CM.png (67 KB)
  Japanese_GPT-4o_t0_Step3_Perception_CM.png (105 KB)
  Japanese_GPT-4o_t0_Step4_Integrated_CM.png (219 KB)
  Japanese_GPT-5_Step1_2_Response_CM.png (70 KB)
  Japanese_GPT-5_Step1_2_Suggestion_CM.png (66 KB)
  Japanese_GPT-5_Step3_Perception_CM.png (104 KB)
  Japanese_GPT-5_Step4_Integrated_CM.png (231 KB)
All PNGs in .\appendix/results/confusion_matrix/:
  American_GPT-4o_t0_Step1_2_Response_CM.png (78 KB)
  American_GPT-4o_t0_Step1_2_Suggestion_CM.png (75 KB)
  American_GPT-4o_t0_Step3_Perception_CM.png (105 KB)
  American_GPT-4o_t0_Step4_Integrated_CM.png (227 KB)
  American_GPT-5_Step1